# Snippet from Math-Constraints-Duality-and-Shadow-Prices.md


There is no `ShadowPriceEstimator`, `project_feasible`, or `check_constraint_qualification` anywhere in `src/compitum` -- shadow prices are NOT computed via a continuous `scipy.optimize.minimize(method='trust-constr')` reimplementation. The real mechanism is `ReflectiveConstraintSolver.select()` in `src/compitum/constraints.py`, and it is a discrete relax-and-recheck scheme.

In [ ]:
from compitum.constraints import ReflectiveConstraintSolver
from compitum.models import Model
from compitum.capabilities import Capabilities
import numpy as np

caps = Capabilities(regions={"US"}, tools_allowed={"none"})
models = [
    Model(name="fast", center=np.zeros(4), capabilities=caps, cost=0.1),
    Model(name="thinking", center=np.zeros(4), capabilities=caps, cost=0.5),
]
A = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=float)
b = np.array([2.0, 2.0, 0.0, 0.0])
solver = ReflectiveConstraintSolver(A, b)

xB = np.array([1.0, 1.0, 0.0, 0.0])
m_star, info = solver.select(xB, models, utilities={"fast": 0.8, "thinking": 0.9})
print("winner:", m_star.name)
print("info:", info)


How it actually works, per row `i` of `A x <= b`: (1) relax `b[i]` by `+1e-5` only, holding every other bound fixed; (2) for every model other than the selected winner, check whether it becomes newly feasible under the relaxed bound *and* has higher utility than the winner; (3) if such a competitor exists, `lambda_i = (utility_competitor - utility_winner) / 1e-5`, otherwise `lambda_i = 0.0`.

**Important, verified caveat**: `A @ xB <= b` does not depend on which model is being checked -- the same boolean result applies to every model, since `xB` is shared. So whenever *any* model is feasible, that linear check already passed for every candidate, and relaxing `b` further cannot retroactively fix a competitor excluded for a different reason (failing `Capabilities.supports()`, which has nothing to do with `b`). In practice this means shadow prices come out `0.0` unless your specific `A`/`b`/capability setup creates a case where relaxing one bound genuinely changes a currently-infeasible-due-to-that-bound competitor status.